## 数据加载器

In [ ]:
## 数据加载
import torch
import torch.nn as nn
from torchvision.datasets import ImageFolder
from torchvision.models import ResNet50_Weights
import torch.utils.data as data
from typing import  Optional

device = torch.device('cuda:0')
print(f"Using device:{device}")


In [ ]:
train_path = r'./datasets/ImageNet/train'
valid_path = r'./datasets/ImageNet/valid'

In [ ]:
weights = ResNet50_Weights.IMAGENET1K_V2
preprocess = weights.transforms()

In [ ]:
train_dataset = ImageFolder(train_path, preprocess)
valid_dataset = ImageFolder(valid_path, preprocess)
batch_size=16

train_data_loader = data.DataLoader(train_dataset,
                                    batch_size=batch_size,
                                    shuffle=True,
                                    num_workers=2)
valid_data_loader = data.DataLoader(valid_dataset,
                                   batch_size=batch_size,
                                   shuffle=False,
                                   num_workers=2)

## 残差神经网络

In [ ]:
## 残差神经网络ResNet50
class ResBlock(nn.Module):
    expansion=4 #resblock属性
    def __init__(self, 
                 in_channels, 
                 hid_channels,
                 stride=1,
                 downsample: Optional[nn.Module] = None):
        super().__init__()
        self.downsample = downsample
        self.relu = nn.SiLU()
        
        
        self.conv1= nn.Conv2d(in_channels, hid_channels, kernel_size=1, bias=False)
        self.bn1= nn.BatchNorm2d(hid_channels)
           
        self.conv2= nn.Conv2d(hid_channels, hid_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2= nn.BatchNorm2d(hid_channels)
       
        self.conv3= nn.Conv2d(hid_channels, hid_channels * self.expansion, kernel_size=1, bias=False)
        self.bn3= nn.BatchNorm2d(hid_channels * self.expansion)
            
    def forward(self, x):
        identity = x
        if self.downsample is not None:
            identity = self.downsample(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)
        
        out += identity 
        out = self.relu(out)
        
        return out
    
class ResNet(nn.Module):
    def __init__(self, block, num_res:list, num_classes):
        super().__init__()
        self.in_channels = 64

        ## 初始层
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False) 
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.SiLU()
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        ## block层
        self.layer1 = self._make_layer(block, 64, num_res[0])
        self.layer2 = self._make_layer(block, 128, num_res[1], 2)
        self.layer3 = self._make_layer(block, 256, num_res[2], 2)
        self.layer4 = self._make_layer(block, 512, num_res[3], 2)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)
        
    def _make_layer(self, block, hid_channels, blocks, stride=1) -> nn.Sequential:
        downsample = None
        if stride != 1 or self.in_channels != hid_channels * block.expansion:
            downsample = nn.Sequential(nn.Conv2d(self.in_channels, hid_channels * block.expansion, kernel_size=1, stride=stride, bias=False),
                                       nn.BatchNorm2d(hid_channels * block.expansion)
                                      )
        layers=[]
        layers.append(block(self.in_channels,hid_channels,stride,downsample))
        
        self.in_channels = hid_channels * block.expansion
        for _ in range(1,blocks):
            layers.append(block(self.in_channels, hid_channels))
            
        return nn.Sequential(*layers)
                
    def forward(self,x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
            
        return x

def resnet50_custom(num_classes= 1000,
                    weights=None, 
                    **kwargs)->ResNet:
    
    original_num_classes = num_classes
    
    if weights is not None: ## 训练类别不等于预训练类别时权重无法匹配，先以预训练类别对卷积层进行训练
        import warnings
        warnings.warn(f"Overwriting{'num_classes'}...")
        kwargs["num_classes"]=len(weights.meta['categories'])

    actual_num_classes = kwargs.get("num_classes",num_classes)

    model = ResNet(ResBlock, [3, 4, 6, 3] ,num_classes=actual_num_classes )

    if weights is not None: ## 使用预训练
        model.load_state_dict(weights.get_state_dict(progress=True,check_hash=True))
    
    if weights is not None and original_num_classes != len(weights.meta['categories']):## 卷积层训练完成后，全连接层恢复原始类别
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        nn.init.xavier_uniform_(model.fc.weight)
        nn.init.zeros_(model.fc.bias)

    return model


In [ ]:
def train(model, dataloader, epochs, lr):
    model.to(device)
    optim = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fun = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        num_batches = 0
        for inputs, labels in dataloader:
            optim.zero_grad()
            pred = model(inputs.to(device))
            loss = loss_fun(pred, labels.to(device))
            loss.backward()
            optim.step()
            total_loss += loss.item()
            num_batches += 1

        avg_loss = total_loss/num_batches
        print(f'epoch{epoch}, Loss:{loss.item():.4f}')  
            

In [ ]:
model = resnet50_custom(num_classes=20, weights=None)
model.to(device)

In [ ]:
train(model, train_data_loader, 10, 1e-3)

In [ ]:
def train_one_epoch(model, dataloader, device):
    model.train()
    total_loss = 0
    num_batches = 0
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
        
    avg_loss = total_loss / num_batches
    return avg_loss
    
def evaluate(model, dataloader,device):
    model.eval()
    total=0
    correct=0
    with torch.no_grad():
        for inputs,labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            pred = torch.argmax(outputs, dim=1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
    accuracy = 100 * correct / total
    print(f'Accuracy on validation set: {accuracy:.2f}%')
    return accuracy  

In [ ]:
import matplotlib.pyplot as plt

train_losses = []
val_accuracies = []

for epoch in range(20):
    loss = train_one_epoch(model, train_data_loader,device)
    acc = evaluate(model, valid_data_loader,device)
    train_losses.append(loss)
    val_accuracies.append(acc)

# 绘图
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.title('Training Loss ')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.subplot(1, 2, 2)
plt.plot(val_accuracies, label='Val Accuracy', color='orange')
plt.title('Validation Accuracy')  
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')

plt.tight_layout()
plt.show()
plt.savefig('training_curve.png', dpi=300, bbox_inches='tight') 
    